# University Placement Analysis

This notebook demonstrates:

- CSV ingestion
- PySpark transformations
- Delta Lake tables
- SQL analytics
- Data visualization

In [0]:
# Add python collection

programs = [
    "MTech CSE",
    "MTech BDA",
    "MTech Cloud Computing",
    "MTech Cyber Security"
]

for program in programs:
    print(program)

MTech CSE
MTech BDA
MTech Cloud Computing
MTech Cyber Security


In [0]:
%sql
-- Create a schema "placement_demo" under workspace catalog

CREATE SCHEMA IF NOT EXISTS workspace.placement_demo
COMMENT 'Schema used for the Databricks placement demonstration';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.placement_demo.source_files
COMMENT 'Volume for CSV source files';

In [0]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
# Create a DataFrame from a list of tuples
df = spark.read.csv("/Volumes/workspace/placement_demo/source_files/university_placements.csv", header=True, inferSchema=True)
df.show()

+----------+------------+------------------+----------+----------+-----------------+-----------+--------------+----------------+----------+
|student_id|student_name|           program|batch_year|   company|         job_role| offer_type|annual_package|placement_status|offer_date|
+----------+------------+------------------+----------+----------+-----------------+-----------+--------------+----------------+----------+
|      S001|  Ananya Rao|            ME CSE|      2026|   Infosys|Software Engineer|  Full-Time|        850000|        Selected|2026-08-10|
|      S002| Rahul Kumar|            ME BDA|      2026|Databricks|    Data Engineer| Internship|       1000000|        Selected|2026-08-12|
|      S003|Meera Shetty|            ME CSE|      2026| Microsoft|Software Engineer|  Full-Time|       1800000|        Selected|2026-08-16|
|      S004|   Arjun Pai|ME Cloud Computing|      2026|       AWS|   Cloud Engineer| Internship|        900000|        Selected|2026-08-18|
|      S005|   Nisha

In [0]:
# Count the records
print("Number of records:", df.count())

Number of records: 15


In [0]:
# Select particular columns
display(
    df.select(
        "student_id",
        "student_name",
        "program",
        "company",
        "annual_package"
    )
)

student_id,student_name,program,company,annual_package
S001,Ananya Rao,ME CSE,Infosys,850000
S002,Rahul Kumar,ME BDA,Databricks,1000000
S003,Meera Shetty,ME CSE,Microsoft,1800000
S004,Arjun Pai,ME Cloud Computing,AWS,900000
S005,Nisha Rao,ME BDA,Accenture,1450000
S006,Vikram Singh,ME Cyber Security,Palo Alto,1200000
S007,Priya Nair,ME CSE,Infosys,850000
S008,Kiran Kumar,ME Cloud Computing,AWS,900000
S009,Asha Bhat,ME BDA,null,0
S010,Rohan Das,ME Cyber Security,null,0


In [0]:
# Filter selected students

from pyspark.sql.functions import col

selected_df = df.filter(
    col("placement_status") == "Selected"
)

display(selected_df)

student_id,student_name,program,batch_year,company,job_role,offer_type,annual_package,placement_status,offer_date
S001,Ananya Rao,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-10
S002,Rahul Kumar,ME BDA,2026,Databricks,Data Engineer,Internship,1000000,Selected,2026-08-12
S003,Meera Shetty,ME CSE,2026,Microsoft,Software Engineer,Full-Time,1800000,Selected,2026-08-16
S004,Arjun Pai,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-18
S005,Nisha Rao,ME BDA,2026,Accenture,Data Engineer,Full-Time,1450000,Selected,2026-08-20
S006,Vikram Singh,ME Cyber Security,2026,Palo Alto,Security Analyst,Full-Time,1200000,Selected,2026-08-22
S007,Priya Nair,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-24
S008,Kiran Kumar,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-25
S011,Divya Rao,ME CSE,2025,TCS,System Engineer,Full-Time,700000,Selected,2025-09-10
S012,Manoj Pai,ME BDA,2025,Databricks,Data Engineer,Full-Time,1700000,Selected,2025-09-15


In [0]:
# Handle NULL values
from pyspark.sql.functions import coalesce, lit

cleaned_df = (
    df
    .withColumn(
        "company",
        coalesce(col("company"), lit("Not Applicable"))
    )
    .withColumn(
        "job_role",
        coalesce(col("job_role"), lit("Not Applicable"))
    )
)

display(cleaned_df)

student_id,student_name,program,batch_year,company,job_role,offer_type,annual_package,placement_status,offer_date
S001,Ananya Rao,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-10
S002,Rahul Kumar,ME BDA,2026,Databricks,Data Engineer,Internship,1000000,Selected,2026-08-12
S003,Meera Shetty,ME CSE,2026,Microsoft,Software Engineer,Full-Time,1800000,Selected,2026-08-16
S004,Arjun Pai,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-18
S005,Nisha Rao,ME BDA,2026,Accenture,Data Engineer,Full-Time,1450000,Selected,2026-08-20
S006,Vikram Singh,ME Cyber Security,2026,Palo Alto,Security Analyst,Full-Time,1200000,Selected,2026-08-22
S007,Priya Nair,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-24
S008,Kiran Kumar,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-25
S009,Asha Bhat,ME BDA,2026,Not Applicable,Not Applicable,Not Offered,0,Not Selected,null
S010,Rohan Das,ME Cyber Security,2026,Not Applicable,Not Applicable,Not Offered,0,Not Selected,null


In [0]:
#Convert the date column
from pyspark.sql.functions import to_date

cleaned_df = cleaned_df.withColumn(
    "offer_date",
    to_date(col("offer_date"), "yyyy-MM-dd")
)

cleaned_df.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- program: string (nullable = true)
 |-- batch_year: integer (nullable = true)
 |-- company: string (nullable = false)
 |-- job_role: string (nullable = false)
 |-- offer_type: string (nullable = true)
 |-- annual_package: integer (nullable = true)
 |-- placement_status: string (nullable = true)
 |-- offer_date: date (nullable = true)



In [0]:
# Create a package category
from pyspark.sql.functions import when

enriched_df = cleaned_df.withColumn(
    "package_category",
    when(col("annual_package") >= 1500000, "Very High")
    .when(col("annual_package") >= 1000000, "High")
    .when(col("annual_package") >= 700000, "Medium")
    .when(col("annual_package") > 0, "Entry Level")
    .otherwise("No Offer")
)

display(enriched_df)

student_id,student_name,program,batch_year,company,job_role,offer_type,annual_package,placement_status,offer_date,package_category
S001,Ananya Rao,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-10,Medium
S002,Rahul Kumar,ME BDA,2026,Databricks,Data Engineer,Internship,1000000,Selected,2026-08-12,High
S003,Meera Shetty,ME CSE,2026,Microsoft,Software Engineer,Full-Time,1800000,Selected,2026-08-16,Very High
S004,Arjun Pai,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-18,Medium
S005,Nisha Rao,ME BDA,2026,Accenture,Data Engineer,Full-Time,1450000,Selected,2026-08-20,High
S006,Vikram Singh,ME Cyber Security,2026,Palo Alto,Security Analyst,Full-Time,1200000,Selected,2026-08-22,High
S007,Priya Nair,ME CSE,2026,Infosys,Software Engineer,Full-Time,850000,Selected,2026-08-24,Medium
S008,Kiran Kumar,ME Cloud Computing,2026,AWS,Cloud Engineer,Internship,900000,Selected,2026-08-25,Medium
S009,Asha Bhat,ME BDA,2026,Not Applicable,Not Applicable,Not Offered,0,Not Selected,null,No Offer
S010,Rohan Das,ME Cyber Security,2026,Not Applicable,Not Applicable,Not Offered,0,Not Selected,null,No Offer



# Create and work with a Delta table

In [0]:

#13
# Save the DataFrame as a Delta table
target_table = "workspace.placement_demo.placement_results"

(
    enriched_df.write
               .format("delta")
               .mode("overwrite")
               .option("overwriteSchema", "true")
               .saveAsTable(target_table)
)